# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [2]:
class Matrix:
    # Cunstructor
    # Matrix(n, m): creates an n x m zero matrix
    # Matrix(list_of_lists): creates matrix from data
    def __init__(self, n, m=None):

        # Initialize by size
        if isinstance(n, int) and isinstance(m, int):
            self.data = [[0.0 for _ in range(m)] for _ in range(n)]

        # Initialize from list of lists
        elif isinstance(n, list):
            if not all(isinstance(row, list) for row in n):
                raise TypeError("Input must be a list of lists")

            row_len = len(n[0])
            for row in n:
                if len(row) != row_len:
                    raise ValueError("All rows must have same length")

            self.data = [[float(x) for x in row] for row in n]

        else:
            raise TypeError("Invalid initialization")

   
# Basic Methods   
    def shape(self):
        return (len(self.data), len(self.data[0]))

    def to_list(self):
        return [row[:] for row in self.data]

    def transpose(self):
        rows, cols = self.shape()
        return Matrix([[self.data[i][j] for i in range(rows)] for j in range(cols)])

    def row(self, n):
        return Matrix([self.data[n][:]])

    def column(self, n):
        return Matrix([[row[n]] for row in self.data])

    # block(n0,n1,m0,m1) → submatrix using slicing
    def block(self, n0, n1, m0, m1):
        return Matrix([row[m0:m1] for row in self.data[n0:n1]])


# Indexing
# Allows BOTH M[i][j] and M[i,j]
    
    def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key
            return self.data[i][j]
        return self.data[key]

    
# Matrix Operation 
    
    def scalarmul(self, c):
        r, c2 = self.shape()
        return Matrix([[self.data[i][j]*c for j in range(c2)] for i in range(r)])

    def add(self, N):
        if self.shape() != N.shape():
            raise ValueError("Addition requires same dimensions")

        r, c = self.shape()
        return Matrix([[self.data[i][j] + N.data[i][j] for j in range(c)] for i in range(r)])

    def sub(self, N):
        if self.shape() != N.shape():
            raise ValueError("Subtraction requires same dimensions")

        r, c = self.shape()
        return Matrix([[self.data[i][j] - N.data[i][j] for j in range(c)] for i in range(r)])

# Standard linear algebra matrix multiplication
    def mat_mult(self, N):
        r1, c1 = self.shape()
        r2, c2 = N.shape()

        if c1 != r2:
            raise ValueError("Inner dimensions must match")

        result = Matrix(r1, c2)

        for i in range(r1):
            for j in range(c2):
                result.data[i][j] = sum(self.data[i][k]*N.data[k][j] for k in range(c1))

        return result

# Element-wise multiplication 
    def element_mult(self, N):
        if self.shape() != N.shape():
            raise ValueError("Element-wise multiplication requires same size")

        r, c = self.shape()
        return Matrix([[self.data[i][j]*N.data[i][j] for j in range(c)] for i in range(r)])

    def equals(self, N):
        return self.data == N.data

# Operator overloading
# Enables natural expressions
   
    def __add__(self, other):
        return self.add(other)

    def __sub__(self, other):
        return self.sub(other)

    def __mul__(self, other):
        if isinstance(other, Matrix):
            return self.mat_mult(other)
        return self.scalarmul(other)

    def __rmul__(self, other):
        return self.scalarmul(other)

    def __eq__(self, other):
        return self.equals(other)

    def __repr__(self):
        return "\n".join(str(row) for row in self.data)



# Special Matrix Functions

def constant(n, m, c):
    return Matrix([[float(c) for _ in range(m)] for _ in range(n)])

def zeros(n, m):
    return constant(n, m, 0.0)

def ones(n, m):
    return constant(n, m, 1.0)

def eye(n):
    I = zeros(n, n)
    for i in range(n):
        I.data[i][i] = 1.0
    return I



# TESTING 

print("----- Initialization -----")
A = Matrix([[1,2],[3,4]])
B = Matrix([[5,6],[7,8]])
C = Matrix([[2,0],[1,2]])
print(A, "\n")

print("----- Shape -----")
print(A.shape(), "\n")

print("----- Indexing -----")
print(A[0][1], A[0,1], "\n")

print("----- Row / Column -----")
print(A.row(0))
print(A.column(1), "\n")

print("----- Block -----")
print(A.block(0,2,0,1), "\n")

print("----- Transpose -----")
print(A.transpose(), "\n")

print("----- Special Matrices -----")
print(zeros(2,2))
print(ones(2,2))
I = eye(2)
print(I, "\n")

print("----- Scalar Multiplication -----")
print(2*A)
print(A*2, "\n")

print("----- Addition / Subtraction -----")
print(A+B)
print(B-A, "\n")

print("----- Matrix Multiplication -----")
print(A*B, "\n")

print("----- Element-wise Multiplication -----")
print(A.element_mult(B), "\n")

print("----- Equality -----")
print(A==A)
print(A==B, "\n")


# DEMONSTRATION

print("----- (AB)C = A(BC) -----")
print((A*B)*C)
print(A*(B*C), "\n")

print("----- A(B+C) = AB + AC -----")
print(A*(B+C))
print((A*B)+(A*C), "\n")

print("----- AB != BA -----")
print("AB:\n", A*B)
print("BA:\n", B*A, "\n")

print("----- AI = A -----")
print(A*I)

----- Initialization -----
[1.0, 2.0]
[3.0, 4.0] 

----- Shape -----
(2, 2) 

----- Indexing -----
2.0 2.0 

----- Row / Column -----
[1.0, 2.0]
[2.0]
[4.0] 

----- Block -----
[1.0]
[3.0] 

----- Transpose -----
[1.0, 3.0]
[2.0, 4.0] 

----- Special Matrices -----
[0.0, 0.0]
[0.0, 0.0]
[1.0, 1.0]
[1.0, 1.0]
[1.0, 0.0]
[0.0, 1.0] 

----- Scalar Multiplication -----
[2.0, 4.0]
[6.0, 8.0]
[2.0, 4.0]
[6.0, 8.0] 

----- Addition / Subtraction -----
[6.0, 8.0]
[10.0, 12.0]
[4.0, 4.0]
[4.0, 4.0] 

----- Matrix Multiplication -----
[19.0, 22.0]
[43.0, 50.0] 

----- Element-wise Multiplication -----
[5.0, 12.0]
[21.0, 32.0] 

----- Equality -----
True
False 

----- (AB)C = A(BC) -----
[60.0, 44.0]
[136.0, 100.0]
[60.0, 44.0]
[136.0, 100.0] 

----- A(B+C) = AB + AC -----
[23.0, 26.0]
[53.0, 58.0]
[23.0, 26.0]
[53.0, 58.0] 

----- AB != BA -----
AB:
 [19.0, 22.0]
[43.0, 50.0]
BA:
 [23.0, 34.0]
[31.0, 46.0] 

----- AI = A -----
[1.0, 2.0]
[3.0, 4.0]
